## Data Sampling with a Sliding Window
This notebook demonstrates how to split a text sample into overlapping input and target blocks for language model training. The model learns to predict the next word for each input block.

In [ ]:
!pip install torch

## Sliding Window Data Sampling
This dataset class splits a sequence of tokens into overlapping input and target blocks using a sliding window. Each block is converted into a PyTorch tensor for direct use in model training.

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## Data Loader for Batching
A data loader is used to generate batches of input and target pairs for efficient model training.

In [10]:
import tiktoken

def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle, drop_last = drop_last, num_workers = num_workers)
    return dataloader

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size = 1, max_length = 4, stride = 1, shuffle = False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch:", first_batch)
second_batch = next(data_iter)
print("Second batch:", second_batch)

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


## Batch Structure and Stride
Each batch contains two tensors: one for input token IDs and one for target token IDs. The max_length parameter sets the size of each block. The stride parameter controls how much the window shifts for each batch—stride=1 means overlapping batches, while stride equal to max_length means no overlap.